In [6]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, make_scorer

from catboost import CatBoostRegressor


# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "data_rdkit.csv"
MODEL_BUNDLE_PATH = "catboost_pipeline_bundle.pkl"

SMILES_COL = "Smiles"
TARGET_COL = "pIC50"

MODEL_PARAMS = dict(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    eval_metric="R2",
    verbose=100,
    random_seed=42
)


# =========================================================
# UTILS
# =========================================================
class ColumnSelector(BaseEstimator, TransformerMixin):

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        missing = [c for c in self.columns if c not in X.columns]
        if missing:
            raise ValueError(f"Missing columns in input DataFrame: {missing}")
        return self

    def transform(self, X):
        return X.loc[:, self.columns]


def rmse_func(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(DATA_PATH)
df = df.dropna().copy()

feature_columns = [c for c in df.columns if c not in [SMILES_COL, TARGET_COL]]

X = df[feature_columns].copy()
y = df[TARGET_COL].values

print(f"Rows: {len(df)}")
print(f"Features: {len(feature_columns)}")
print("Feature columns:")
print(feature_columns)


from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


df["scaffold"] = df[SMILES_COL].apply(get_scaffold)

print("Unique scaffolds:", df["scaffold"].nunique())
from sklearn.model_selection import GroupKFold


# =========================================================
# TRAIN / TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# =========================================================
# PIPELINE
# =========================================================
pipeline = Pipeline([
    ("select_columns", ColumnSelector(feature_columns)),
    ("model", CatBoostRegressor(**MODEL_PARAMS))
])


# =========================================================
# CROSS-VALIDATION
# =========================================================
print("=" * 50)
print("RANDOM CV")
print("=" * 50)

cv_r2_random = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print(f"Random CV R2 mean: {cv_r2_random.mean():.4f}")
print(f"Random CV R2 std:  {cv_r2_random.std():.4f}")


print("\n" + "=" * 50)
print("SCAFFOLD CV")
print("=" * 50)

gkf = GroupKFold(n_splits=5)

cv_r2_scaffold = cross_val_score(
    pipeline,
    X,
    y,
    groups=df["scaffold"],
    cv=gkf,
    scoring="r2"
)

print(f"Scaffold CV R2 mean: {cv_r2_scaffold.mean():.4f}")
print(f"Scaffold CV R2 std:  {cv_r2_scaffold.std():.4f}")

# =========================================================
# FIT
# =========================================================
pipeline.fit(X_train, y_train)


# =========================================================
# EVALUATION
# =========================================================
y_pred = pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = rmse_func(y_test, y_pred)

print("\n" + "=" * 50)
print("TEST METRICS")
print("=" * 50)
print(f"R2 Score: {r2:.4f}")
print(f"RMSE:     {rmse:.4f}")


# =========================================================
# FEATURE IMPORTANCE
# =========================================================
model = pipeline.named_steps["model"]
importances = model.get_feature_importance()

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\nFeature importance:")
print(importance_df)


# =========================================================
# SAVE BUNDLE
# =========================================================
bundle = {
    "pipeline": pipeline,
    "feature_columns": feature_columns,
    "smiles_col": SMILES_COL,
    "target_col": TARGET_COL,
    "model_params": MODEL_PARAMS,
    "logic_note": "Features are all columns except Smiles and pIC50"
}

with open(MODEL_BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"\nSaved bundle to: {MODEL_BUNDLE_PATH}")

Rows: 12358
Features: 164
Feature columns:
['MaxAbsEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'MaxPartialCharge', 'MinPartialCharge', 'FpDensityMorgan1', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'AvgIpc', 'BalabanJ', 'BertzCT', 'HallKierAlpha', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'SMR_VSA1', 'SMR_VSA10', 'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7', 'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA10', 'SlogP_VSA11', 'SlogP_VSA12', 'SlogP_VSA2', 'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA7', 'SlogP_VSA8', 'TPSA', 'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 'EState_VSA9', 'VSA_EState1', 'VSA_

In [7]:
print(cv_r2_scaffold)

[0.66187737 0.6657247  0.63808397 0.6088448  0.55008657]


In [8]:
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import cross_val_score, GroupKFold, KFold
from sklearn.metrics import make_scorer, mean_squared_error
from catboost import CatBoostRegressor


# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "data_rdkit.csv"

SMILES_COL = "Smiles"
TARGET_COL = "pIC50"

FP_RADIUS = 3
FP_NBITS = 1024
FP_PREFIX = "fp_"

MODEL_PARAMS = dict(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    eval_metric="R2",
    verbose=0,
    random_seed=42
)


# =========================================================
# HELPERS
# =========================================================
def rmse_func(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "INVALID"
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


def morgan_fp_bits(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [0] * n_bits
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.tolist()


def build_pipeline(desc_cols, fp_cols, model_params):
    transformers = []

    if len(desc_cols) > 0:
        desc_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        transformers.append(("desc", desc_pipe, desc_cols))

    if len(fp_cols) > 0:
        fp_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("variance", VarianceThreshold(threshold=0.0))
        ])
        transformers.append(("fp", fp_pipe, fp_cols))

    preprocessor = ColumnTransformer(transformers, remainder="drop")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", CatBoostRegressor(**model_params))
    ])
    return pipeline


def evaluate_mode(X, y, groups, desc_cols, fp_cols, mode_name):
    pipeline = build_pipeline(desc_cols, fp_cols, MODEL_PARAMS)

    random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scaffold_cv = GroupKFold(n_splits=5)

    random_r2 = cross_val_score(
        pipeline,
        X,
        y,
        cv=random_cv,
        scoring="r2",
        n_jobs=-1
    )

    scaffold_r2 = cross_val_score(
        pipeline,
        X,
        y,
        groups=groups,
        cv=scaffold_cv,
        scoring="r2",
        n_jobs=-1
    )

    rmse_scorer = make_scorer(
        lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
        greater_is_better=False
    )

    random_rmse = -cross_val_score(
        pipeline,
        X,
        y,
        cv=random_cv,
        scoring=rmse_scorer,
        n_jobs=-1
    )

    scaffold_rmse = -cross_val_score(
        pipeline,
        X,
        y,
        groups=groups,
        cv=scaffold_cv,
        scoring=rmse_scorer,
        n_jobs=-1
    )

    return {
        "mode": mode_name,
        "n_desc": len(desc_cols),
        "n_fp": len(fp_cols),
        "n_total": len(desc_cols) + len(fp_cols),
        "random_r2_mean": random_r2.mean(),
        "random_r2_std": random_r2.std(),
        "scaffold_r2_mean": scaffold_r2.mean(),
        "scaffold_r2_std": scaffold_r2.std(),
        "random_rmse_mean": random_rmse.mean(),
        "random_rmse_std": random_rmse.std(),
        "scaffold_rmse_mean": scaffold_rmse.mean(),
        "scaffold_rmse_std": scaffold_rmse.std(),
        "random_r2_folds": random_r2,
        "scaffold_r2_folds": scaffold_r2
    }


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(DATA_PATH).dropna().copy()

print(f"Initial rows: {len(df)}")

# scaffold groups
df["scaffold"] = df[SMILES_COL].apply(get_scaffold)

# descriptors already present in CSV
descriptor_columns = [
    c for c in df.columns
    if c not in [SMILES_COL, TARGET_COL, "scaffold"]
]

print(f"Descriptor features found in CSV: {len(descriptor_columns)}")

# =========================================================
# CALCULATE FINGERPRINTS
# =========================================================
print("Calculating Morgan fingerprints...")

fp_matrix = df[SMILES_COL].apply(
    lambda smi: morgan_fp_bits(smi, radius=FP_RADIUS, n_bits=FP_NBITS)
)

fp_df = pd.DataFrame(
    fp_matrix.tolist(),
    columns=[f"{FP_PREFIX}{i}" for i in range(FP_NBITS)],
    index=df.index
)

df = pd.concat([df, fp_df], axis=1)

fingerprint_columns = list(fp_df.columns)

print(f"Fingerprint features added: {len(fingerprint_columns)}")
print(f"Unique scaffolds: {df['scaffold'].nunique()}")
print(df["scaffold"].value_counts().describe())


# =========================================================
# PREPARE MATRICES
# =========================================================
all_feature_columns = descriptor_columns + fingerprint_columns

X = df[all_feature_columns].copy()
y = df[TARGET_COL].values
groups = df["scaffold"].values


# =========================================================
# RUN 3 EXPERIMENTS
# =========================================================
results = []

experiments = [
    ("descriptors_only", descriptor_columns, []),
    ("fingerprints_only", [], fingerprint_columns),
    ("descriptors_plus_fingerprints", descriptor_columns, fingerprint_columns),
]

for mode_name, desc_cols, fp_cols in experiments:
    print("\n" + "=" * 70)
    print(f"Running: {mode_name}")
    print("=" * 70)

    res = evaluate_mode(
        X=X,
        y=y,
        groups=groups,
        desc_cols=desc_cols,
        fp_cols=fp_cols,
        mode_name=mode_name
    )
    results.append(res)

    print(f"Random CV R2:    {res['random_r2_mean']:.4f} ± {res['random_r2_std']:.4f}")
    print(f"Scaffold CV R2:  {res['scaffold_r2_mean']:.4f} ± {res['scaffold_r2_std']:.4f}")
    print(f"Random CV RMSE:  {res['random_rmse_mean']:.4f} ± {res['random_rmse_std']:.4f}")
    print(f"Scaffold CV RMSE:{res['scaffold_rmse_mean']:.4f} ± {res['scaffold_rmse_std']:.4f}")
    print(f"Random R2 folds:   {np.round(res['random_r2_folds'], 4)}")
    print(f"Scaffold R2 folds: {np.round(res['scaffold_r2_folds'], 4)}")


# =========================================================
# SUMMARY TABLE
# =========================================================
results_df = pd.DataFrame(results)[[
    "mode",
    "n_desc",
    "n_fp",
    "n_total",
    "random_r2_mean",
    "random_r2_std",
    "scaffold_r2_mean",
    "scaffold_r2_std",
    "random_rmse_mean",
    "random_rmse_std",
    "scaffold_rmse_mean",
    "scaffold_rmse_std"
]].sort_values("scaffold_r2_mean", ascending=False)

print("\n" + "=" * 70)
print("FINAL COMPARISON")
print("=" * 70)
print(results_df.to_string(index=False))

Initial rows: 12358
Descriptor features found in CSV: 164
Calculating Morgan fingerprints...


[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerator
[10:44:08] DEPRECATION WARNING: please use MorganGenerat

Fingerprint features added: 1024
Unique scaffolds: 4735
count    4735.000000
mean        2.609926
std         7.393270
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       272.000000
Name: count, dtype: float64

Running: descriptors_only
Random CV R2:    0.7600 ± 0.0134
Scaffold CV R2:  0.6249 ± 0.0426
Random CV RMSE:  0.7141 ± 0.0206
Scaffold CV RMSE:0.8908 ± 0.1113
Random R2 folds:   [0.7649 0.7762 0.7474 0.7412 0.7702]
Scaffold R2 folds: [0.6619 0.6657 0.6381 0.6088 0.5501]

Running: fingerprints_only
Random CV R2:    0.7688 ± 0.0056
Scaffold CV R2:  0.6396 ± 0.0474
Random CV RMSE:  0.7011 ± 0.0094
Scaffold CV RMSE:0.8727 ± 0.1134
Random R2 folds:   [0.767  0.7758 0.7662 0.7607 0.7741]
Scaffold R2 folds: [0.6638 0.7064 0.6473 0.6155 0.565 ]

Running: descriptors_plus_fingerprints
Random CV R2:    0.7812 ± 0.0125
Scaffold CV R2:  0.6626 ± 0.0411
Random CV RMSE:  0.6818 ± 0.0192
Scaffold CV RMSE:0.8446 ± 0.1068
Random R2 folds:   [0.7832 0.7962

In [9]:
import pickle
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GroupKFold

from catboost import CatBoostRegressor


# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "data_rdkit.csv"
ENSEMBLE_BUNDLE_PATH = "catboost_ensemble_bundle.pkl"

SMILES_COL = "Smiles"
TARGET_COL = "pIC50"

FP_RADIUS = 2
FP_NBITS = 2048
FP_PREFIX = "fp_"

N_ENSEMBLE_MODELS = 5

MODEL_PARAMS = dict(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    eval_metric="R2",
    verbose=0,
    random_seed=42
)


# =========================================================
# HELPERS
# =========================================================
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "INVALID"
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


def morgan_fp_bits(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [0] * n_bits
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.tolist()


def morgan_fp_bitvect(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)


def build_pipeline(desc_cols, fp_cols, model_params):
    transformers = []

    if len(desc_cols) > 0:
        desc_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        transformers.append(("desc", desc_pipe, desc_cols))

    if len(fp_cols) > 0:
        fp_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("variance", VarianceThreshold(threshold=0.0))
        ])
        transformers.append(("fp", fp_pipe, fp_cols))

    preprocessor = ColumnTransformer(transformers, remainder="drop")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", CatBoostRegressor(**model_params))
    ])
    return pipeline

In [10]:
# =========================================================
# TRAIN ENSEMBLE
# =========================================================
base_pipeline = build_pipeline(
    descriptor_columns,
    fingerprint_columns,
    MODEL_PARAMS
)

gkf = GroupKFold(n_splits=N_ENSEMBLE_MODELS)

ensemble_models = []
fold_info = []

for fold_id, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    print(f"Training ensemble model {fold_id}/{N_ENSEMBLE_MODELS} ...")

    X_train_fold = X.iloc[train_idx]
    y_train_fold = y[train_idx]

    model = clone(base_pipeline)
    model.fit(X_train_fold, y_train_fold)

    ensemble_models.append(model)
    fold_info.append({
        "fold_id": fold_id,
        "train_size": len(train_idx),
        "valid_size": len(valid_idx)
    })

print("Ensemble training done.")

Training ensemble model 1/5 ...
Training ensemble model 2/5 ...
Training ensemble model 3/5 ...
Training ensemble model 4/5 ...
Training ensemble model 5/5 ...
Ensemble training done.


In [ ]:
# =========================================================
# STORE TRAIN FINGERPRINTS FOR SIMILARITY-BASED UNCERTAINTY
# =========================================================
train_bitvects = [
    morgan_fp_bitvect(smi, radius=FP_RADIUS, n_bits=FP_NBITS)
    for smi in df[SMILES_COL]
]

bundle = {
    "ensemble_models": ensemble_models,
    "descriptor_columns": descriptor_columns,
    "fingerprint_columns": fingerprint_columns,
    "smiles_col": SMILES_COL,
    "target_col": TARGET_COL,
    "fp_radius": FP_RADIUS,
    "fp_nbits": FP_NBITS,
    "train_smiles": df[SMILES_COL].tolist(),
    "train_bitvects": train_bitvects,
    "fold_info": fold_info,
    "model_params": MODEL_PARAMS
}

with open(ENSEMBLE_BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"Saved ensemble bundle to: {ENSEMBLE_BUNDLE_PATH}")

[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerator
[11:02:48] DEPRECATION WARNING: please use MorganGenerat

In [ ]:
def featurize_smiles_to_row(smiles, descriptor_columns, fp_radius, fp_nbits, fp_prefix="fp_"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    row = {}

    # descriptors
    from rdkit.Chem import Descriptors, QED
    registry = {name: func for name, func in Descriptors.descList}
    registry["qed"] = QED.qed
    registry["QED"] = QED.qed

    for col in descriptor_columns:
        func = registry.get(col)
        if func is None:
            raise KeyError(f"Descriptor '{col}' not found in RDKit registry.")
        try:
            val = func(mol)
            if val is None or not np.isfinite(val):
                val = 0.0
        except Exception:
            val = 0.0
        row[col] = float(val)

    # fingerprints
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, fp_radius, nBits=fp_nbits)
    arr = np.zeros((fp_nbits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    for i in range(fp_nbits):
        row[f"{fp_prefix}{i}"] = int(arr[i])

    return row


def max_tanimoto_similarity(query_bv, train_bitvects):
    sims = []
    for bv in train_bitvects:
        if bv is None:
            continue
        sims.append(DataStructs.TanimotoSimilarity(query_bv, bv))
    return max(sims) if sims else 0.0


def predict_with_uncertainty(smiles_list, bundle):
    descriptor_columns = bundle["descriptor_columns"]
    fp_radius = bundle["fp_radius"]
    fp_nbits = bundle["fp_nbits"]
    train_bitvects = bundle["train_bitvects"]
    ensemble_models = bundle["ensemble_models"]

    rows = []
    valid_mask = []
    query_bitvects = []

    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rows.append(None)
            valid_mask.append(False)
            query_bitvects.append(None)
            continue

        row = featurize_smiles_to_row(
            smi,
            descriptor_columns=descriptor_columns,
            fp_radius=fp_radius,
            fp_nbits=fp_nbits
        )
        rows.append(row)
        valid_mask.append(True)
        query_bitvects.append(
            AllChem.GetMorganFingerprintAsBitVect(mol, fp_radius, nBits=fp_nbits)
        )

    # build dataframe
    valid_rows = []
    valid_indices = []

    for i, row in enumerate(rows):
        if row is not None:
            valid_rows.append(row)
            valid_indices.append(i)

    results = []
    if valid_rows:
        Xq = pd.DataFrame(valid_rows)

        all_preds = []
        for model in ensemble_models:
            preds = model.predict(Xq)
            all_preds.append(preds)

        all_preds = np.vstack(all_preds)              # [n_models, n_samples]
        pred_mean = all_preds.mean(axis=0)
        pred_std = all_preds.std(axis=0)

        pred_map = {}
        for idx, mean_val, std_val in zip(valid_indices, pred_mean, pred_std):
            pred_map[idx] = (float(mean_val), float(std_val))

    for i, smi in enumerate(smiles_list):
        if not valid_mask[i]:
            results.append({
                "smiles": smi,
                "pred_mean": 0.0,
                "pred_std": 999.0,
                "max_similarity": 0.0,
                "novelty": 1.0,
                "is_valid": False
            })
            continue

        mean_val, std_val = pred_map[i]
        sim = max_tanimoto_similarity(query_bitvects[i], train_bitvects)

        results.append({
            "smiles": smi,
            "pred_mean": mean_val,
            "pred_std": std_val,
            "max_similarity": float(sim),
            "novelty": float(1.0 - sim),
            "is_valid": True
        })

    return pd.DataFrame(results)

In [ ]:
with open("catboost_ensemble_bundle.pkl", "rb") as f:
    bundle = pickle.load(f)

test_smiles = [
    "CCO",
    "c1ccccc1",
    "CC(=O)Oc1ccccc1C(=O)O"
]

pred_df = predict_with_uncertainty(test_smiles, bundle)
print(pred_df)

In [ ]:
smiles = [
    "CCO",
    "c1ccccc1",
    "CC(=O)Oc1ccccc1C(=O)O",
]

with open(r"C:\Users\norep\Desktop\hakaton\test_smiles.smi", "w", encoding="utf-8", newline="\n") as f:
    f.write("\n".join(smiles))